# Earthquake Intensity LSTM — Leakage-Free Corrected Notebook

**Purpose:** corrected, leakage-free version of the original capstone model.

**Dataset:** `earthquake_dataset.csv`

The held-out test set is isolated before any data-dependent preprocessing.  
For time-series models, chronological splitting is performed **before** scaling and sliding-window construction.  
Validation data are used for early stopping/model selection; the test set is evaluated only after training is complete.

In [ ]:
# Install only if needed in Colab
!pip -q install pandas numpy matplotlib scikit-learn tensorflow

import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
DATA_FILE = "earthquake_dataset.csv"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("Please upload", DATA_FILE)
        uploaded = files.upload()
    except Exception as e:
        raise FileNotFoundError(DATA_FILE + " was not found.") from e

assert os.path.exists(DATA_FILE), "Missing file: " + DATA_FILE

In [ ]:
df = pd.read_csv(DATA_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
invalid_ts = int(df["timestamp"].isna().sum())
df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

df["vibration_detected"] = (
    df["vibration_detected"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
)

LABEL_MAP = {"Low": 0, "Medium": 1, "High": 2, "Critical": 2}
df["intensity"] = df["risk_level"].map(LABEL_MAP)

FEATURES = ["motion", "vibration_detected"]
SEQUENCE_LENGTH = 15
CLASS_NAMES = ["Low", "Medium", "High"]

df = df.dropna(subset=FEATURES + ["intensity"]).reset_index(drop=True)

print("Rows after cleaning:", len(df))
print("Invalid timestamps removed:", invalid_ts)
print("Overall 3-class counts:")
print(df["intensity"].value_counts().sort_index())

In [ ]:
n = len(df)
train_end = int(0.64 * n)
val_end = int(0.80 * n)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

scaler = MinMaxScaler()
scaler.fit(train_df[FEATURES])

def build_sequences(part_df, sequence_length=SEQUENCE_LENGTH):
    X_scaled = scaler.transform(part_df[FEATURES])
    y = part_df["intensity"].astype(int).to_numpy()

    X_seq, y_seq = [], []
    for i in range(sequence_length, len(part_df)):
        X_seq.append(X_scaled[i-sequence_length:i])
        y_seq.append(y[i])

    return np.asarray(X_seq, dtype=np.float32), np.asarray(y_seq, dtype=np.int32)

X_train, y_train = build_sequences(train_df)
X_val,   y_val   = build_sequences(val_df)
X_test,  y_test  = build_sequences(test_df)

def show_counts(name, y):
    vals, counts = np.unique(y, return_counts=True)
    print(name, {CLASS_NAMES[int(v)]: int(c) for v, c in zip(vals, counts)})

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
show_counts("Train", y_train)
show_counts("Validation", y_val)
show_counts("Test", y_test)

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, len(FEATURES))),
    LSTM(32, return_sequences=False, unroll=True),
    Dropout(0.2),
    Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(prob, axis=1)

acc = accuracy_score(y_test, y_pred)
prec_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy         : {acc:.6f}")
print(f"Macro precision  : {prec_macro:.6f}")
print(f"Macro recall     : {rec_macro:.6f}")
print(f"Macro F1         : {f1_macro:.6f}")
print(f"Weighted F1      : {f1_weighted:.6f}")
print()
print(classification_report(
    y_test, y_pred,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES,
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(values_format="d")
plt.title("Earthquake Intensity — Leakage-Free Test Set")
plt.show()

pd.DataFrame({
    "true_class": [CLASS_NAMES[i] for i in y_test],
    "predicted_class": [CLASS_NAMES[i] for i in y_pred]
}).to_csv("earthquake_leakage_free_test_predictions.csv", index=False)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Earthquake LSTM Training")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
model.save("earthquake_intensity_lstm_leakage_free.keras")

meta = {
    "features": FEATURES,
    "sequence_length": SEQUENCE_LENGTH,
    "class_names": CLASS_NAMES,
    "label_map": LABEL_MAP,
    "scaler_min": scaler.min_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "split": {"train": 0.64, "validation": 0.16, "test": 0.20},
    "seed": SEED
}
with open("earthquake_intensity_lstm_leakage_free_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
converter._experimental_lower_tensor_list_ops = True
converter.experimental_enable_resource_variables = False
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("earthquake_intensity_lstm_leakage_free.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved corrected earthquake artifacts.")